# Trading Bot – Full Pipeline

**Anleitung:**
1. Accelerator auf **GPU T4 x2** stellen (Settings → Accelerator)
2. Dataset **busersteven/trading-raw-data** hinzufügen (Add data → Your datasets)
3. In der Python-Zelle die zwei Schalter oben einstellen (siehe unten)
4. **Run All** oder **Save Version → Save & Run All (Commit)**

---

### Schalter (in der Python-Zelle ganz oben einstellen)

| Variable | Wert | Bedeutung |
|---|---|---|
| `SMOKE_TEST` | `True` | **Schnelltest** – 15 Assets, 3 Epochen, ~2 Folds → fertig in ~10–15 Min |
| `SMOKE_TEST` | `False` | **Echter Run** – alle ~260 Assets, 50 Epochen, 12 Folds → ~2–3 Std |
| `HORIZONS` | `[7]` | Nur 7-Tage-Horizont trainieren |
| `HORIZONS` | `[4, 7, 11, 15]` | Alle vier Horizonte |

**Empfehlung:** erst `SMOKE_TEST = True` laufen lassen – wenn alles grün, dann `False` für den echten Run.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  SCHALTER  ←  hier einstellen, dann Run All
# ═══════════════════════════════════════════════════════════════
SMOKE_TEST = False  # True = Schnelltest (~10-15 Min) | False = echter Run (~2-3h)
HORIZONS   = [7]    # Horizonte in Tagen, z.B. [7] oder [4, 7, 11, 15]
SEED       = 42     # Random Seed fuer reproduzierbares Training (42, 1, 7, ...)
# ═══════════════════════════════════════════════════════════════

import os
import re
import subprocess
import sys
import time

# Schalter als Umgebungsvariablen setzen (werden von kaggle_full_run.py gelesen)
os.environ["KAGGLE_SH_HORIZONS"]  = ",".join(str(h) for h in HORIZONS)
os.environ["KAGGLE_SMOKE_TEST"]   = "1" if SMOKE_TEST else "0"
os.environ["KAGGLE_SEED"]         = str(SEED)
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")

print("=" * 55)
print(f"  Modus     : {'*** SMOKE-TEST (~10-15 Min) ***' if SMOKE_TEST else 'ECHTER RUN  (~2-3 Stunden)'}")
print(f"  Horizonte : {HORIZONS}")
print(f"  Seed      : {SEED}")
if SMOKE_TEST:
    print("  Assets    : 15  | Epochen: 3  | Folds: ~2")
else:
    print("  Assets    : ~260 | Epochen: 50 | Folds: 12")
print("=" * 55)

# Pipeline-Code von GitHub laden (Cache-Buster verhindert veraltete Version)
cache_bust = int(time.time())
url = (
    f"https://raw.githubusercontent.com/stevenlangeshops/trading/main/"
    f"scripts/kaggle_full_run.py?cb={cache_bust}"
)
r = subprocess.run(
    ["wget", "-q", "-O", "/kaggle/working/kaggle_full_run.py", url],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(r.stdout or "download ok")

path = "/kaggle/working/kaggle_full_run.py"
with open(path, encoding="utf-8") as f:
    code = f.read()

# Fallback fuer aeltere Script-Versionen mit fester SH_HORIZONS-Zeile
code, n_legacy = re.subn(
    r"^(\s*SH_HORIZONS\s*=\s*)\[\s*\d+\s*,\s*\d+\s*\]",
    lambda m: m.group(1) + repr(HORIZONS),
    code,
    flags=re.MULTILINE,
)
if n_legacy:
    print(f"[legacy] {n_legacy} SH_HORIZONS-Zeile(n) auf {HORIZONS!r} gesetzt")

# Modul-Cache leeren (verhindert Stale-Code bei Re-Runs)
for mod in list(sys.modules.keys()):
    if mod.startswith(("strategy", "models", "features", "config_v2", "train_v2", "backtest_v2")):
        del sys.modules[mod]

exec(code)